# RA Cohort Comparison: ICD-Only Baseline vs MAP Pipeline

Compares two approaches to identifying RA patients in MIMIC-IV:

| Method | Description | Signal |
|--------|-------------|--------|
| **ICD baseline** | Patients with ≥`MIN_ANCHOR_ENCOUNTERS` ICD encounters mapping to PheCode 714.1 | ICD codes only |
| **MAP pipeline** | Patients assigned phenotype=1 by MAP mixture model | ICD + ONCE co-features |

`INCLUDE_NLP = True` — MAP uses both codified features (PheCodes + RxNorm) and NLP (CUI mentions from discharge notes).

## What to expect from this comparison

**Without NLP, MAP is a quality filter, not a case finder.** Because MAP's anchor is
`PheCode:714.1` (derived from ICD codes), a patient with zero RA ICD coding has
anchor = 0 and will almost never be classified as a case by the mixture model.
The sensitivity gain from MAP requires NLP (discharge note CUI mentions let MAP
flag patients who were under-coded in ICD).

**What MAP *does* provide without NLP — higher specificity:**
- MAP fits a Poisson mixture model over all ONCE co-features jointly.
  A patient with *one* stray RA ICD code but no supporting co-features
  (no DMARDs, no related joint PheCodes) gets a low posterior score and is
  classified as a control.
- The ICD baseline accepts all such patients as cases.

**The key claim to verify here:**
> MAP cases (phenotype=1) should be more enriched for RA clinical correlates
> (DMARD prescriptions, steroids, RA comorbidities) than ICD cases,
> because MAP removed the lowest-evidence ICD codings.

The `icd_only` group (ICD=case, MAP=control) represents probable ICD false positives.
The `map_only` group (MAP=case, ICD=control) should be near zero without NLP.

In [1]:
import os
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
SAMPLE_N = None  # None = full MIMIC-IV; set e.g. 50_000 for speed
RANDOM_SEED = 42
MAIN_PHECODE = "714.1"
ANCHOR_EVENT = f"PheCode:{MAIN_PHECODE}"
INCLUDE_NLP = True
MAX_NOTE_CHARS = 10000  # truncate each note before NLP for speed
MIN_ANCHOR_ENCOUNTERS = 2  # minimum anchor PheCode events to count as ICD case
NOTES_PER_PATIENT = 1  # most recent discharge notes to use per patient for NLP

# ── Clinical validation keywords ──────────────────────────────────────────────
DMARD_KEYWORDS = [
    "methotrexate",
    "hydroxychloroquine",
    "plaquenil",
    "leflunomide",
    "arava",
    "sulfasalazine",
    "adalimumab",
    "humira",
    "etanercept",
    "enbrel",
    "infliximab",
    "remicade",
    "rituximab",
    "abatacept",
    "orencia",
    "tocilizumab",
    "actemra",
    "tofacitinib",
    "xeljanz",
    "baricitinib",
]
STEROID_KEYWORDS = [
    "prednisone",
    "methylprednisolone",
    "prednisolone",
    "dexamethasone",
]
COMORBIDITY_PHECODES = {
    "Sjogren's syndrome": "710.2",
    "ILD / pulm fibrosis": "516.3",
    "Osteoporosis": "743.1",
    "Chronic anemia": "285",
    "Vasculitis": "446",
}

# ── Paths ─────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = os.path.abspath(".")
PROJECT_DIR = os.path.abspath("..")
MIMIC_HOSP = os.path.join(PROJECT_DIR, "mimiciv", "hosp")
NOTES_FILE = os.path.join(PROJECT_DIR, "mimiciv", "note", "discharge.csv.gz")

PHECODE_MAP = os.path.join(NOTEBOOK_DIR, "Phecode_map_v1_2_icd9_icd10cm.csv")
ONCE_CODIFIED = os.path.join(
    NOTEBOOK_DIR, "ONCE_Rheumatoid Arthritis_PheCode714.1_cos0.165.csv"
)
ONCE_NARRATIVE = os.path.join(
    NOTEBOOK_DIR,
    "ONCE_Rheumatoid Arthritis_C0003873_titlecos0.5_titlecut0.3_exactFALSE.csv",
)

for _p in [
    NOTEBOOK_DIR,
    os.path.join(NOTEBOOK_DIR, "map"),
    os.path.join(NOTEBOOK_DIR, "preprocessing"),
]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

for label, path in [
    ("MIMIC hosp", MIMIC_HOSP),
    ("PheCode map", PHECODE_MAP),
    ("ONCE codified", ONCE_CODIFIED),
    ("Notes file", NOTES_FILE),
]:
    print(f"  [{'OK' if os.path.exists(path) else 'MISSING'}] {label}")

if INCLUDE_NLP and not os.path.exists(NOTES_FILE):
    print("\nWARNING: notes file missing — setting INCLUDE_NLP = False")
    INCLUDE_NLP = False

  [OK] MIMIC hosp
  [OK] PheCode map
  [OK] ONCE codified
  [OK] Notes file


## 1. Load Data

In [2]:
# Sample patients
patients_df = pd.read_csv(
    os.path.join(MIMIC_HOSP, "patients.csv.gz"), usecols=["subject_id"]
)
if SAMPLE_N and SAMPLE_N < len(patients_df):
    sampled_ids = set(
        patients_df["subject_id"].sample(SAMPLE_N, random_state=RANDOM_SEED)
    )
    print(f"Sampled {SAMPLE_N:,} / {len(patients_df):,} patients")
else:
    sampled_ids = set(patients_df["subject_id"])
    print(f"Using all {len(sampled_ids):,} patients")

# Admissions (note count proxy for MAP's Poisson denominator)
admissions_df = pd.read_csv(
    os.path.join(MIMIC_HOSP, "admissions.csv.gz"),
    usecols=["subject_id", "hadm_id", "admittime"],
    parse_dates=["admittime"],
)
admissions_df = admissions_df[admissions_df["subject_id"].isin(sampled_ids)]
print(
    f"Admissions : {len(admissions_df):,} rows | {admissions_df['subject_id'].nunique():,} patients"
)

# Diagnoses joined with admissions for event dates
diagnoses_df = pd.read_csv(
    os.path.join(MIMIC_HOSP, "diagnoses_icd.csv.gz"),
    usecols=["subject_id", "hadm_id", "icd_code", "icd_version"],
    dtype={"icd_code": str},
)
diagnoses_df = diagnoses_df[diagnoses_df["subject_id"].isin(sampled_ids)]
diagnoses_with_dates = diagnoses_df.merge(
    admissions_df[["hadm_id", "admittime"]], on="hadm_id", how="left"
)
print(
    f"Diagnoses  : {len(diagnoses_df):,} rows | {diagnoses_df['subject_id'].nunique():,} patients"
)

Using all 364,627 patients
Admissions : 546,028 rows | 223,452 patients
Diagnoses  : 6,364,488 rows | 223,291 patients


In [3]:
from once import get_once_features, parse_once_by_modality
from preprocessing import icd_to_events

# Load ONCE features
once_features = get_once_features(ONCE_CODIFIED, ONCE_NARRATIVE)
by_modality = parse_once_by_modality(once_features)
once_phecodes = set(by_modality["phecode"])  # bare codes like '714.1'
print(
    f"ONCE codified: {len(once_features['codified_list'])} features "
    f"({len(by_modality['phecode'])} PheCodes, {len(by_modality['rxnorm'])} RxNorm, "
    f"{len(by_modality['shortname'])} shortname)"
)

# Filter comorbidity PheCodes to those NOT in ONCE features (held-out validators)
EXTERNAL_COMORBIDITIES = {
    label: code
    for label, code in COMORBIDITY_PHECODES.items()
    if code not in once_phecodes
}
if EXTERNAL_COMORBIDITIES:
    print(
        f"\nExternal comorbidity validators (not in ONCE): {list(EXTERNAL_COMORBIDITIES.keys())}"
    )
else:
    print("\n[WARN] All comorbidity codes in ONCE — using all anyway.")
    EXTERNAL_COMORBIDITIES = COMORBIDITY_PHECODES

# Build ICD observation log (shared by both methods)
icd_obs = icd_to_events(
    diagnoses_with_dates,
    icd_col="icd_code",
    date_col="admittime",
    mapping_file=PHECODE_MAP,
)
print(
    f"\nICD obs log: {len(icd_obs):,} rows | {icd_obs['subject_id'].nunique():,} patients "
    f"| {icd_obs['event'].nunique()} unique PheCodes"
)

ONCE codified: 41 features (28 PheCodes, 10 RxNorm, 3 shortname)

External comorbidity validators (not in ONCE): ['ILD / pulm fibrosis', 'Osteoporosis', 'Chronic anemia', 'Vasculitis']

ICD obs log: 5,943,457 rows | 222,960 patients | 1785 unique PheCodes


In [ ]:
nlp_obs = pd.DataFrame(
    columns=["subject_id", "event_type", "event", "value", "datetime"]
)

if INCLUDE_NLP:
    from preprocessing import notes_to_events

    once_events = set(once_features["codified_list"])
    nlp_candidate_ids = set(icd_obs[icd_obs["event"].isin(once_events)]["subject_id"])
    print(f"NLP candidates : {len(nlp_candidate_ids):,} patients")

    note_chunks = []
    for chunk in pd.read_csv(
        NOTES_FILE,
        usecols=["subject_id", "text", "charttime"],
        parse_dates=["charttime"],
        chunksize=5_000,
    ):
        sub = chunk[chunk["subject_id"].isin(nlp_candidate_ids)].copy()
        if len(sub):
            note_chunks.append(sub)

    notes_df = pd.concat(note_chunks, ignore_index=True).dropna(subset=["text"])
    print(f"Total notes available  : {len(notes_df):,}")
    print(f"Notes per patient cap  : {NOTES_PER_PATIENT} most recent")

    nlp_obs = notes_to_events(
        notes_df,
        text_col="text",
        date_col="charttime",
        target_cuis=once_features["nlp_target_cuis"],
        max_note_chars=MAX_NOTE_CHARS,
        notes_per_patient=NOTES_PER_PATIENT,
    )
    print(
        f"NLP obs log            : {len(nlp_obs):,} CUI mentions | "
        f"{nlp_obs['subject_id'].nunique():,} patients"
    )
else:
    print("INCLUDE_NLP=False — NLP step skipped.")

NLP candidates : 12,476 patients
Total notes available  : 41,683
Notes per patient cap  : 1 most recent


  MedSpaCy:  10%|█         | 1047/10320 [02:39<24:10,  6.39note/s]

## 2. Cohort 1 — ICD Baseline

**Rule:** any patient with ≥1 ICD code mapping to PheCode 714.1 (RA anchor) is a case.
This is the classical rule-based approach an agent without MAP skills would use.

In [ ]:
# ICD baseline: patients with >= MIN_ANCHOR_ENCOUNTERS anchor PheCode events
anchor_counts = icd_obs[icd_obs["event"] == ANCHOR_EVENT].groupby("subject_id").size()
icd_case_ids = set(anchor_counts[anchor_counts >= MIN_ANCHOR_ENCOUNTERS].index)
icd_ctrl_ids = set(icd_obs["subject_id"]) - icd_case_ids

print("=== ICD Baseline ===")
print(f"Cases (≥{MIN_ANCHOR_ENCOUNTERS} anchor ICD encounters): {len(icd_case_ids):,}")
print(f"Controls (< {MIN_ANCHOR_ENCOUNTERS} anchor ICD):         {len(icd_ctrl_ids):,}")
print(f"Total patients w/ ICD:                    {icd_obs['subject_id'].nunique():,}")
print(
    f"Case rate:                                {100 * len(icd_case_ids) / icd_obs['subject_id'].nunique():.1f}%"
)

print("\nAnchor PheCode encounter distribution (ICD cases only):")
print(
    anchor_counts[anchor_counts >= MIN_ANCHOR_ENCOUNTERS]
    .value_counts()
    .sort_index()
    .head(10)
    .to_string()
)

## 3. Cohort 2 — MAP Pipeline

MAP fits a Poisson mixture model over the ONCE-selected co-feature count matrix
(anchor PheCode + joint-disease PheCodes + DMARD RxNorm codes).

**With NLP disabled, MAP's advantage over ICD is specificity, not sensitivity:**
- Patients with one stray RA code but no co-features → MAP scores them near 0 (control)
- Patients with many RA encounters AND co-features → MAP scores them near 1 (case)
- `map_only` cases (MAP=1 without an ICD anchor) should be ~0%

The ONCE features available to MAP in this run include both PheCodes and RxNorm
DMARD codes — so MAP leverages prescription patterns that the raw ICD filter ignores entirely.

In [ ]:
from map import preprocess_map, run_map

obs_log = pd.concat([icd_obs, nlp_obs], ignore_index=True)
print(
    f"obs_log : {len(obs_log):,} rows  "
    f"({obs_log['event_type'].value_counts().to_dict()})"
)

mat_df, note_df = preprocess_map(
    obs_log=obs_log,
    admissions_df=admissions_df,
    once_features=once_features,
    main_phecode=MAIN_PHECODE,
)
print(f"\nmat_df : {mat_df.shape}  (candidates × ONCE features)")
print(f"Features retained: {list(mat_df.columns)}")

print("\nRunning MAP...")
map_results = run_map(mat_df, note_df, main_icd_col=ANCHOR_EVENT)

map_case_ids = set(map_results[map_results["phenotype"] == 1]["patient_id"])
map_ctrl_ids = set(map_results[map_results["phenotype"] == 0]["patient_id"])

print("\n=== MAP Results ===")
print(f"Candidates total:       {len(map_results):,}")
print(f"Cases    (phenotype=1): {len(map_case_ids):,}")
print(f"Controls (phenotype=0): {len(map_ctrl_ids):,}")
print(f"Case rate:              {100 * len(map_case_ids) / len(map_results):.1f}%")

## 4. Cohort Overlap

Both methods are evaluated within the MAP candidate population (patients with ≥1 ONCE feature).

**Expected result without NLP:**
- `icd_only` > 0: patients ICD coded but MAP rejected → the false positives MAP cleaned out
- `map_only` ≈ 0: MAP cannot add cases the ICD rule missed without NLP signal
- MAP case rate ≤ ICD case rate (MAP is a subset of ICD, not a superset)

**This is correct behaviour**, not a failure. With NLP enabled, `map_only` would grow:
those are patients mentioned as having RA in discharge notes but under-coded in ICD.

In [ ]:
# Align ICD baseline to MAP's candidate population
candidate_ids = set(mat_df.index)
icd_case_cand = icd_case_ids & candidate_ids
icd_ctrl_cand = candidate_ids - icd_case_cand

# Four groups
both_case = map_case_ids & icd_case_cand  # MAP=case, ICD=case  (agreed)
icd_only = icd_case_cand - map_case_ids  # MAP=ctrl, ICD=case  (MAP rejected)
map_only = map_case_ids - icd_case_cand  # MAP=case, ICD=ctrl  (MAP found)
not_either = candidate_ids - map_case_ids - icd_case_cand  # both control

N = len(candidate_ids)
print("=== Cohort overlap within candidates ===")
print(f"{'Candidates total':38s}: {N:>5,}  (100.0%)")
print(
    f"{'Both agree: case':38s}: {len(both_case):>5,}  ({100 * len(both_case) / N:.1f}%)"
)
print(
    f"{'ICD case but MAP rejected':38s}: {len(icd_only):>5,}  ({100 * len(icd_only) / N:.1f}%)  ← likely false positives"
)
print(
    f"{'MAP case not in ICD filter':38s}: {len(map_only):>5,}  ({100 * len(map_only) / N:.1f}%)  ← coded RA missed by ICD rule"
)
print(
    f"{'Both agree: control':38s}: {len(not_either):>5,}  ({100 * len(not_either) / N:.1f}%)"
)

print("\n")
summary_df = pd.DataFrame(
    {
        "Cohort": ["ICD Baseline", "MAP Pipeline"],
        "Cases": [len(icd_case_cand), len(map_case_ids)],
        "Controls": [len(icd_ctrl_cand), len(map_ctrl_ids)],
        "Case rate (%)": [
            round(100 * len(icd_case_cand) / N, 1),
            round(100 * len(map_case_ids) / N, 1),
        ],
    }
)
display(summary_df.set_index("Cohort"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: stacked bar showing composition of each method's cases
categories = ["Both agree (case)", "ICD only", "MAP only"]
icd_vals = [len(both_case), len(icd_only), 0]
map_vals = [len(both_case), 0, len(map_only)]
x = np.arange(len(categories))
axes[0].bar(
    ["ICD cases", "MAP cases"],
    [len(icd_case_cand), len(map_case_ids)],
    color=["#FF9800", "#2196F3"],
    edgecolor="white",
    linewidth=1.2,
)
for rect, val in zip(axes[0].patches, [len(icd_case_cand), len(map_case_ids)]):
    axes[0].text(
        rect.get_x() + rect.get_width() / 2,
        rect.get_height() + 0.5,
        str(val),
        ha="center",
        fontweight="bold",
        fontsize=12,
    )
axes[0].set_title("Case Count per Method", fontsize=12)
axes[0].set_ylabel("Patients")
axes[0].set_ylim(0, max(len(icd_case_cand), len(map_case_ids)) * 1.2)

# Right: four-group breakdown
labels = [
    "Both cases",
    "ICD only\n(MAP rejected)",
    "MAP only\n(missed by ICD)",
    "Both controls",
]
sizes = [len(both_case), len(icd_only), len(map_only), len(not_either)]
colors = ["#4CAF50", "#FF9800", "#2196F3", "#9E9E9E"]
bars = axes[1].bar(labels, sizes, color=colors, edgecolor="white", linewidth=1.2)
for bar, size in zip(bars, sizes):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        str(size),
        ha="center",
        fontweight="bold",
        fontsize=10,
    )
axes[1].set_title("Candidate Assignment Breakdown", fontsize=12)
axes[1].set_ylabel("Patients")
axes[1].set_ylim(0, max(sizes) * 1.15)

plt.tight_layout()
plt.show()

## 5. Clinical Validation Signals

Load prescription data to flag each patient with DMARD/biologic and corticosteroid use.
These are the strongest real-world markers of a true RA diagnosis and are **external** to
the MAP model input — providing an honest out-of-model validation.

RA comorbidities from the ICD obs log are also used, filtered to PheCodes not in ONCE features.

In [ ]:
print("Loading prescriptions (this may take ~30 sec)...")
rx_df = pd.read_csv(
    os.path.join(MIMIC_HOSP, "prescriptions.csv.gz"),
    usecols=["subject_id", "drug"],
    dtype={"drug": str},
)
rx_df = rx_df[rx_df["subject_id"].isin(sampled_ids)]
rx_lower = rx_df["drug"].str.lower().fillna("")
print(
    f"Prescriptions: {len(rx_df):,} rows | {rx_df['subject_id'].nunique():,} patients"
)


def pts_with_keywords(rx_df, rx_lower, keywords):
    mask = rx_lower.str.contains("|".join(keywords), case=False, na=False)
    return set(rx_df.loc[mask, "subject_id"])


dmard_pts = pts_with_keywords(rx_df, rx_lower, DMARD_KEYWORDS)
steroid_pts = pts_with_keywords(rx_df, rx_lower, STEROID_KEYWORDS)
print(f"\nPatients with ≥1 DMARD/biologic prescription:  {len(dmard_pts):,}")
print(f"Patients with ≥1 corticosteroid prescription:   {len(steroid_pts):,}")

## 6. Enrichment Analysis

Compare the four cohort groups (ICD cases, MAP cases, ICD controls, MAP controls)
across all clinical validation signals. A method is better if its **cases** show
higher enrichment AND its **controls** show lower enrichment — i.e., greater signal lift.

In [ ]:
def prevalence(cohort_ids, signal_ids):
    if not cohort_ids:
        return 0.0
    return len(cohort_ids & signal_ids) / len(cohort_ids)


# Build signal dictionary
signal_sets = {
    "DMARD / biologic": dmard_pts,
    "Corticosteroid": steroid_pts,
}
for label, phecode in EXTERNAL_COMORBIDITIES.items():
    pts = set(icd_obs[icd_obs["event"] == f"PheCode:{phecode}"]["subject_id"])
    signal_sets[label] = pts

cohorts = {
    "ICD cases": icd_case_cand,
    "MAP cases": map_case_ids,
    "ICD controls": icd_ctrl_cand,
    "MAP controls": map_ctrl_ids,
}

rows = []
for signal_name, signal_ids in signal_sets.items():
    row = {"Signal": signal_name}
    for cname, cids in cohorts.items():
        row[cname] = f"{100 * prevalence(cids, signal_ids):.1f}%"
    rows.append(row)

enrich_df = pd.DataFrame(rows).set_index("Signal")
print("=== Clinical Signal Prevalence ===")
display(enrich_df)

# Lift = case rate / control rate
print("\n=== Signal lift (case prevalence / control prevalence) ===")
for signal_name, signal_ids in signal_sets.items():
    icd_lift = prevalence(icd_case_cand, signal_ids) / max(
        prevalence(icd_ctrl_cand, signal_ids), 0.001
    )
    map_lift = prevalence(map_case_ids, signal_ids) / max(
        prevalence(map_ctrl_ids, signal_ids), 0.001
    )
    winner = "MAP" if map_lift > icd_lift else "ICD"
    print(
        f"  {signal_name:25s}  ICD lift: {icd_lift:5.1f}x   MAP lift: {map_lift:5.1f}x   [{winner} higher]"
    )

In [ ]:
# Numeric version for plotting
rows_num = []
for signal_name, signal_ids in signal_sets.items():
    row = {"Signal": signal_name}
    for cname, cids in cohorts.items():
        row[cname] = 100 * prevalence(cids, signal_ids)
    rows_num.append(row)
enrich_num = pd.DataFrame(rows_num).set_index("Signal")

n_signals = len(enrich_num)
x = np.arange(n_signals)
width = 0.35

fig, ax = plt.subplots(figsize=(max(10, n_signals * 2.2), 5))
bars1 = ax.bar(
    x - width / 2,
    enrich_num["ICD cases"],
    width,
    label="ICD cases",
    color="#FF9800",
    alpha=0.85,
    edgecolor="white",
)
bars2 = ax.bar(
    x + width / 2,
    enrich_num["MAP cases"],
    width,
    label="MAP cases",
    color="#2196F3",
    alpha=0.85,
    edgecolor="white",
)
ax.plot(
    x - width / 2,
    enrich_num["ICD controls"],
    "o--",
    color="#FF9800",
    alpha=0.45,
    label="ICD controls",
    markersize=6,
    linewidth=1.5,
)
ax.plot(
    x + width / 2,
    enrich_num["MAP controls"],
    "s--",
    color="#2196F3",
    alpha=0.45,
    label="MAP controls",
    markersize=6,
    linewidth=1.5,
)

ax.set_xticks(x)
ax.set_xticklabels(enrich_num.index, rotation=25, ha="right", fontsize=10)
ax.set_ylabel("Prevalence (%)")
ax.set_title(
    "Clinical Signal Prevalence: ICD Baseline vs MAP Pipeline\n"
    "(bars = cases; dashed lines = controls; higher bar + lower dashed = better method)",
    fontsize=11,
)
ax.legend()
ax.set_ylim(0, enrich_num.max().max() * 1.35)
plt.tight_layout()
plt.show()

## 7. MAP Score Distribution and Stratification

MAP produces a continuous posterior probability — something the ICD baseline cannot.
A high-quality MAP run on RA should show:
- A **bimodal score distribution** (clear cases near 1, clear controls near 0)
- **Monotonically increasing** DMARD/steroid prevalence as the score quartile increases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Score histogram
axes[0].hist(
    map_results["score"], bins=40, color="#2196F3", edgecolor="white", alpha=0.85
)
med = map_results["score"].median()
axes[0].axvline(
    med, color="red", linestyle="--", linewidth=1.5, label=f"median={med:.3f}"
)
axes[0].set_title("MAP Score Distribution", fontsize=12)
axes[0].set_xlabel("Posterior probability")
axes[0].set_ylabel("Patients")
axes[0].legend()

# Score by phenotype label
axes[1].boxplot(
    [
        map_results.loc[map_results["phenotype"] == 0, "score"].values,
        map_results.loc[map_results["phenotype"] == 1, "score"].values,
    ],
    labels=[
        f"Controls\n(n={len(map_ctrl_ids):,})",
        f"Cases\n(n={len(map_case_ids):,})",
    ],
    patch_artist=True,
    boxprops=dict(facecolor="#E3F2FD"),
)
axes[1].set_title("Score by MAP Phenotype Label", fontsize=12)
axes[1].set_ylabel("MAP posterior score")

plt.tight_layout()
plt.show()

low = (map_results["score"] < 0.1).mean()
high = (map_results["score"] > 0.9).mean()
mid = 1 - low - high
print(f"Score < 0.1 (clear controls): {low:.1%}")
print(f"Score > 0.9 (clear cases):    {high:.1%}")
print(f"Score 0.1–0.9 (uncertain):    {mid:.1%}")
if mid > 0.30:
    print(
        "[WARN] >30% uncertain scores — try a larger SAMPLE_N for more co-feature coverage."
    )
else:
    print("[OK] Score distribution is reasonably bimodal.")

In [ ]:
scored = map_results.set_index("patient_id").copy()

# Compute quartiles; if all scores are identical, fall back to two groups
try:
    scored["quartile"] = pd.qcut(
        scored["score"],
        q=4,
        labels=["Q1 (low)", "Q2", "Q3", "Q4 (high)"],
        duplicates="drop",
    )
except ValueError:
    scored["quartile"] = pd.cut(scored["score"], bins=2, labels=["Low", "High"])

q_labels = scored["quartile"].cat.categories.tolist()
q_sizes = scored.groupby("quartile").size()

# Use first two signals (DMARD + steroid) for the quartile plot
signal_names_q = list(signal_sets.keys())[:2]

rows_q = []
for signal_name in signal_names_q:
    row = {"Signal": signal_name}
    for q in q_labels:
        q_pts = set(scored[scored["quartile"] == q].index)
        row[q] = 100 * prevalence(q_pts, signal_sets[signal_name])
    rows_q.append(row)
quartile_df = pd.DataFrame(rows_q).set_index("Signal")

print("=== Clinical signal prevalence by MAP score quartile ===")
display(quartile_df.round(1).astype(str) + "%")

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
x_q = np.arange(len(q_labels))
for i, signal_name in enumerate(signal_names_q):
    offset = (i - len(signal_names_q) / 2 + 0.5) * 0.38
    ax.bar(
        x_q + offset,
        quartile_df.loc[signal_name, q_labels],
        width=0.35,
        label=signal_name,
        alpha=0.85,
    )

ax.set_xticks(x_q)
ax.set_xticklabels([f"{q}\n(n={q_sizes[q]:,})" for q in q_labels])
ax.set_ylabel("Prevalence (%)")
ax.set_title(
    "Clinical Signal Prevalence by MAP Score Quartile\n"
    "(monotonic increase from Q1→Q4 validates that MAP score tracks clinical severity)",
    fontsize=11,
)
ax.legend()
plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print("=" * 65)
print("COMPARISON SUMMARY: ICD Baseline vs MAP Pipeline")
print("=" * 65)
print(f"\nStudy population : {len(sampled_ids):,} MIMIC-IV patients")
print(f"Candidates       : {len(candidate_ids):,} (≥1 ONCE feature)")
print(f"\n{'Method':<22}  {'Cases':>7}  {'Controls':>9}  {'Case rate':>10}")
print(f"{'-' * 52}")
print(
    f"{'ICD Baseline':<22}  {len(icd_case_cand):>7,}  {len(icd_ctrl_cand):>9,}  "
    f"{100 * len(icd_case_cand) / len(candidate_ids):>9.1f}%"
)
print(
    f"{'MAP Pipeline':<22}  {len(map_case_ids):>7,}  {len(map_ctrl_ids):>9,}  "
    f"{100 * len(map_case_ids) / len(candidate_ids):>9.1f}%"
)

# MAP vs ICD difference
rejected = len(icd_only)
extra = len(map_only)
print("\n--- Overlap ---")
print(
    f"ICD cases MAP rejected (probable false positives) : {rejected:,}  "
    f"({100 * rejected / max(len(icd_case_cand), 1):.1f}% of ICD cases)"
)
print(
    f"MAP cases not in ICD filter (expected ~0 w/o NLP) : {extra:,}  "
    f"({100 * extra / max(len(map_case_ids), 1):.1f}% of MAP cases)"
)

print("\n--- Interpretation ---")
if rejected > 0:
    print(f"MAP removed {rejected:,} patients the ICD rule accepted. These are likely")
    print("miscoded or single-encounter RA patients with no supporting co-features.")
if extra == 0 or extra < 0.02 * len(map_case_ids):
    print(f"map_only ≈ {extra:,} — as expected with NLP disabled.")
    print("Enable INCLUDE_NLP=True to see MAP's sensitivity gain from discharge notes.")

print("\n--- Enrichment (does MAP add clinical coherence?) ---")
for signal_name in list(signal_sets.keys())[:2]:
    signal_ids = signal_sets[signal_name]
    icd_r = 100 * prevalence(icd_case_cand, signal_ids)
    map_r = 100 * prevalence(map_case_ids, signal_ids)
    ctrl_r = 100 * prevalence(map_ctrl_ids, signal_ids)
    delta = map_r - icd_r
    print(f"\n  {signal_name}:")
    print(
        f"    ICD cases: {icd_r:.1f}%   MAP cases: {map_r:.1f}%   MAP controls: {ctrl_r:.1f}%"
    )
    if delta > 0:
        print(f"    → MAP cohort is {delta:.1f} pp more enriched than ICD cohort ✓")
        print("       (MAP removed the lowest-signal ICD cases)")
    else:
        print(
            f"    → ICD cohort more enriched by {-delta:.1f} pp — check co-feature coverage"
        )